# Concert Med Ops — Unsloth QLoRA Fine-Tune

Fine-tunes a Gemma 4 instruction-tuned model on the Concert Med Ops harm-reduction knowledge base to create a domain-specific toxicology / festival-medicine assistant.

The resulting GGUF is drop-in compatible with Ollama via `MODEL_MEDICAL=concert-med-tox:latest`.

## Environment

| Requirement | Min spec | Recommended |
|---|---|---|
| GPU VRAM | 8 GB (4-bit) | 16 GB (4-bit) |
| GPU | T4 (free Kaggle/Colab) | L4 / A10G |
| Python | 3.10+ | 3.11 |
| CUDA | 11.8+ | 12.1 |

## Key documentation

| Resource | URL |
|---|---|
| Unsloth homepage | https://unsloth.ai/ |
| Unsloth docs (main) | https://docs.unsloth.ai/ |
| Fine-tuning guide | https://docs.unsloth.ai/get-started/fine-tuning-guide |
| Dataset format guide | https://docs.unsloth.ai/basics/datasets-guide |
| ShareGPT format spec | https://docs.unsloth.ai/basics/datasets-guide#conversational |
| Saving models guide | https://docs.unsloth.ai/basics/saving-models |
| Exporting to Ollama | https://docs.unsloth.ai/basics/saving-models#ollama |
| Unsloth GitHub | https://github.com/unslothai/unsloth |
| Example notebooks | https://github.com/unslothai/unsloth-zoo/tree/main/notebooks |
| HF TRL SFTTrainer | https://huggingface.co/docs/trl/sft_trainer |
| HF PEFT LoRA | https://huggingface.co/docs/peft/conceptual_guides/lora |
| Gemma 4 model card | https://huggingface.co/google/gemma-4-E2B-it |
| Unsloth Gemma 4 4B (4-bit) | https://huggingface.co/unsloth/gemma-4-E2B-it-unsloth-bnb-4bit |

## Steps

1. Install Unsloth
2. Load base model (Gemma 4 4B IT, 4-bit quantised)
3. Generate training data from the KB
4. Configure QLoRA adapters
5. Train with SFTTrainer
6. Evaluate sample outputs
7. Export to GGUF (Q4_K_M) for Ollama
8. (Optional) Push to Hugging Face Hub

## 1. Install Unsloth

Unsloth requires a matching CUDA version. The command below auto-detects your CUDA version.

Ref: https://docs.unsloth.ai/get-started/installing-+-updating-unsloth

> **Kaggle / Colab users**: Run `!nvidia-smi` first to confirm your GPU type, then use the appropriate install command from the docs above.

In [ ]:
# Install Unsloth — choose the correct variant for your CUDA version:
# Ref: https://docs.unsloth.ai/get-started/installing-+-updating-unsloth
#
# CUDA 12.1  (T4 on Kaggle, most Colab Pro+ GPUs)
# !pip install unsloth
#
# CUDA 11.8  (older T4 configurations)
# !pip install "unsloth[cu118-torch240] @ git+https://github.com/unslothai/unsloth.git"
#
# CPU-only / development (very slow, not recommended for training):
# !pip install "unsloth[colab-new]"
#
# When Unsloth adds Gemma 4 support, pin the version that first ships it:
# !pip install "unsloth>=2025.xx"

!pip install unsloth datasets transformers trl peft accelerate bitsandbytes --quiet

In [ ]:
# Verify GPU is available and check VRAM
import torch

assert torch.cuda.is_available(), "No GPU found — switch to a GPU runtime."
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name}  |  VRAM: {vram_gb:.1f} GB")

if vram_gb < 8:
    print("⚠️  < 8 GB VRAM — reduce per_device_train_batch_size to 1 and gradient_accumulation_steps to 8.")

## 2. Load the Base Model

We use **Gemma 4 4B IT** (instruction-tuned) pre-quantised to 4-bit via bitsandbytes.

- HuggingFace model card: https://huggingface.co/google/gemma-4-E2B-it
- Unsloth 4-bit version: https://huggingface.co/unsloth/gemma-4-E2B-it-unsloth-bnb-4bit
- Unsloth `FastLanguageModel` API: https://docs.unsloth.ai/get-started/fine-tuning-guide#2-load-the-model

### Why Gemma 4 4B?

The production app uses `gemma4:e2b` (Gemma 4 2B edge) in Ollama. Gemma 4 4B is the nearest
Unsloth-supported model with:
- Comparable inference cost on CPU/GPU
- Existing Unsloth-optimised weights on HuggingFace
- Proven quality on medical instruction-following tasks

Once Unsloth publishes `unsloth/gemma-4-*-bnb-4bit`, swap the `model_name` below.

### `max_seq_length`

Set to 2048 — long enough to hold a full clinical encounter + protocol excerpts.
Increase to 4096 if you see truncated RAG context in training examples (costs more VRAM).

In [ ]:
from unsloth import FastLanguageModel

# Model selection
# Ref: https://docs.unsloth.ai/get-started/all-our-models
# - unsloth/gemma-4-E2B-it-unsloth-bnb-4bit   ← default for this notebook (8 GB VRAM)
# - unsloth/gemma-4-E4B-it-unsloth-bnb-4bit  ← higher quality, needs 16 GB VRAM
# - unsloth/gemma-4-E2B-it-unsloth-bnb-4bit   ← ultra-light, lowest quality, < 6 GB VRAM
MODEL_NAME = "unsloth/gemma-4-E2B-it-unsloth-bnb-4bit"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,          # None = auto-detect: bfloat16 on Ampere+, float16 on older
    load_in_4bit=True,   # 4-bit QLoRA — mandatory for T4/L4 with 4B model
)

print(f"Loaded: {MODEL_NAME}")
print(f"Tokenizer vocab size: {tokenizer.tokenizer.vocab_size if hasattr(tokenizer, 'tokenizer') else tokenizer.vocab_size:,}")

## 3. Generate Training Data

Run `generate_training_data.py` to convert `backend/data/harm_reduction_kb.json` into
a JSONL dataset in ShareGPT format.

Ref: https://docs.unsloth.ai/basics/datasets-guide#conversational

In [ ]:
import pathlib, sys

REPO_ROOT = pathlib.Path.cwd().parent  # adjust if running from a different CWD
NOTEBOOKS_DIR = REPO_ROOT / "notebooks"
KB_PATH = REPO_ROOT / "backend" / "data" / "harm_reduction_kb.json"
DATASET_PATH = NOTEBOOKS_DIR / "training_data.jsonl"

assert KB_PATH.exists(), f"KB not found: {KB_PATH}"
sys.path.insert(0, str(NOTEBOOKS_DIR))

from generate_training_data import build_dataset, write_jsonl

examples = build_dataset(kb_path=KB_PATH, fmt="sharegpt", shuffle_seed=42)
write_jsonl(examples, DATASET_PATH)

print(f"Generated {len(examples)} training examples → {DATASET_PATH}")

In [ ]:
from datasets import load_dataset

# Load the JSONL as a HuggingFace Dataset
# Ref: https://huggingface.co/docs/datasets/loading#json-files
raw_dataset = load_dataset("json", data_files=str(DATASET_PATH), split="train")

print(f"Dataset size: {len(raw_dataset)} rows")
print("\nSample conversations field:")
sample = raw_dataset[0]["conversations"]
for turn in sample:
    print(f"  [{turn['from']}]: {str(turn['value'])[:120]}...")

### Apply the Gemma 4 chat template

Gemma 4 uses a specific chat format with `<start_of_turn>` / `<end_of_turn>` tokens.
Unsloth's `get_chat_template` handles this automatically.

Ref: https://docs.unsloth.ai/basics/chat-templates  
Ref: https://huggingface.co/google/gemma-4-E2B-it#chat-template

In [ ]:
from unsloth.chat_templates import get_chat_template, standardize_sharegpt

# Apply Gemma 4 chat template to the tokenizer
# Ref: https://docs.unsloth.ai/basics/chat-templates#gemma
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

# Standardise ShareGPT format (normalises 'from' field aliases: 'assistant' → 'gpt' etc.)
# Ref: https://docs.unsloth.ai/basics/datasets-guide#conversational
dataset = standardize_sharegpt(raw_dataset)

def apply_template(examples):
    """Tokenise conversations using the Gemma 4 chat template."""
    texts = tokenizer.apply_chat_template(
        examples["conversations"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": texts}

dataset = dataset.map(apply_template, batched=True)

# Sanity-check: print the first formatted example
print("=== Formatted example (first 600 chars) ===")
print(dataset[0]["text"][:600])
print("...")

## 4. Configure QLoRA Adapters

We use **QLoRA** (Quantised Low-Rank Adaptation) — fine-tuning a small set of adapter
weights on top of the frozen 4-bit quantised base model.

### Parameter guide

| Parameter | Value | Why |
|---|---|---|
| `r` | 16 | Rank of LoRA matrices — higher = more capacity, more VRAM |
| `lora_alpha` | 16 | Scaling factor; `lora_alpha/r = 1.0` is a safe default |
| `lora_dropout` | 0 | Unsloth recommends 0 for speed; add 0.05 if overfitting |
| `target_modules` | all linear layers | Gemma 4 attention + MLP layers |
| `use_rslora` | False | RSLoRA stabilises very high ranks (r > 64); not needed here |
| `use_gradient_checkpointing` | "unsloth" | Unsloth's vRAM-saving variant (recommended) |

Refs:
- Unsloth LoRA config: https://docs.unsloth.ai/get-started/fine-tuning-guide#3-add-lora-adapters
- LoRA original paper: https://arxiv.org/abs/2106.09685
- RSLoRA paper: https://arxiv.org/abs/2312.03732

In [ ]:
# Add QLoRA adapters
# Ref: https://docs.unsloth.ai/get-started/fine-tuning-guide#3-add-lora-adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    # All attention projection matrices + MLP gate/up/down in Gemma 4
    # Ref: https://huggingface.co/google/gemma-4-E2B-it/blob/main/config.json
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Unsloth's memory-efficient variant
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

# Print trainable parameter count
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

## 5. Train with SFTTrainer

Uses HuggingFace TRL's `SFTTrainer` (Supervised Fine-Tuning Trainer) with Unsloth's
patched training loop for 2–4× faster throughput.

### Training duration guide

| `max_steps` | Time (T4) | Use for |
|---|---|---|
| 60 | ~5 min | Quick sanity check |
| 200 | ~20 min | Development iteration |
| `num_train_epochs=3` | ~1 hr | Production fine-tune |

Set `max_steps=None` and `num_train_epochs=3` for a full production run.

Refs:
- SFTTrainer docs: https://huggingface.co/docs/trl/sft_trainer
- TrainingArguments: https://huggingface.co/docs/transformers/main_classes/trainer#transformers.TrainingArguments
- Unsloth training guide: https://docs.unsloth.ai/get-started/fine-tuning-guide#4-train-the-model

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# ─── Hyperparameters ──────────────────────────────────────────────────────────
# For a quick sanity-check, keep MAX_STEPS small.
# For a production fine-tune: set MAX_STEPS = None and NUM_EPOCHS = 3.
MAX_STEPS = 60      # set to None for full training
NUM_EPOCHS = 1      # ignored when MAX_STEPS is set
BATCH_SIZE = 2      # reduce to 1 if OOM on T4 with 8 GB VRAM
GRAD_ACCUM = 4      # effective batch = BATCH_SIZE * GRAD_ACCUM = 8
LR = 2e-4
OUTPUT_DIR = "./concert-med-tox-checkpoints"

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer.tokenizer if hasattr(tokenizer, 'tokenizer') else tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,  # True can improve throughput for very short sequences
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=5,
        max_steps=MAX_STEPS,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LR,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",     # 8-bit Adam from bitsandbytes — saves VRAM
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir=OUTPUT_DIR,
        report_to="none",       # set to "wandb" if you want W&B logging
    ),
)

print(f"Training on {len(dataset)} examples | MAX_STEPS={MAX_STEPS} | LR={LR}")

In [ ]:
# Log GPU memory before training
gpu_stats = torch.cuda.get_device_properties(0)
reserved = torch.cuda.memory_reserved() / 1e9
print(f"GPU: {gpu_stats.name} | VRAM: {gpu_stats.total_memory / 1e9:.1f} GB")
print(f"Reserved before training: {reserved:.2f} GB")

# Train
train_stats = trainer.train()

# Log GPU memory after training
used = torch.cuda.max_memory_reserved() / 1e9
print(f"Peak VRAM used during training: {used:.2f} GB")
print(f"Training loss (final step): {train_stats.training_loss:.4f}")

## 6. Evaluate — Sample Inference

Run a few clinical prompts through the fine-tuned model to sanity-check quality
before committing to export.

Ref: https://docs.unsloth.ai/get-started/fine-tuning-guide#5-inference

In [ ]:
from unsloth.chat_templates import get_chat_template

# Switch model to inference mode (disables gradient checkpointing)
FastLanguageModel.for_inference(model)

EVAL_PROMPTS = [
    "A 22-year-old on sertraline presents with temperature 104°F, clonus, and agitation after taking MDMA. Is this serotonin syndrome or heat stroke? What do I do first?",
    "Patient is unresponsive with pinpoint pupils and RR of 5. Friends say he snorted something. Walk me through naloxone dosing.",
    "An MDMA user drank lots of water but is now confused and has a headache. Should I give IV fluids?",
]

SYSTEM_PROMPT = (
    "You are Concert Med Ops AI, a clinical decision support assistant for supervising physicians, "
    "paramedics, and harm reduction volunteers at live music events and EDM festivals. "
    "Provide rapid, actionable guidance. For life-threatening emergencies, activate EMS immediately. "
    "\n\n⚠️ DISCLAIMER: AI-generated guidance. Not a substitute for licensed medical care."
)

def run_inference(prompt: str, max_new_tokens: int = 512) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    outputs = model.generate(
        input_ids=input_ids,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
        use_cache=True,
    )
    # Decode only the newly generated tokens
    generated = outputs[0][input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


for i, prompt in enumerate(EVAL_PROMPTS, 1):
    print(f"\n{'='*70}")
    print(f"EVAL {i}: {prompt}")
    print("-" * 70)
    response = run_inference(prompt)
    print(response)

## 7. Export to GGUF for Ollama

Export the fine-tuned LoRA model merged into a GGUF file for local serving with Ollama.

### Quantisation method guide

| Method | Size (4B model) | Quality | Use when |
|---|---|---|---|
| `q4_k_m` | ~2.5 GB | ★★★★☆ | Default — best quality/size tradeoff |
| `q5_k_m` | ~3.0 GB | ★★★★★ | More VRAM available |
| `q8_0` | ~4.5 GB | ★★★★★ | Near-lossless, large disk |
| `f16` | ~8.0 GB | ★★★★★ | Dev / evaluation only |

Refs:
- Unsloth saving guide: https://docs.unsloth.ai/basics/saving-models
- Unsloth → Ollama workflow: https://docs.unsloth.ai/basics/saving-models#ollama
- llama.cpp quantisation types: https://github.com/ggerganov/llama.cpp/blob/master/docs/gguf-spec.md

In [ ]:
# Return to training mode temporarily to merge weights
FastLanguageModel.for_training(model)

GGUF_OUTPUT_DIR = "concert-med-tox"
QUANT_METHOD = "q4_k_m"   # See table above — q4_k_m is the recommended default

# Export: merges LoRA weights into the base model and quantises to GGUF
# Ref: https://docs.unsloth.ai/basics/saving-models#gguf
model.save_pretrained_gguf(
    GGUF_OUTPUT_DIR,
    tokenizer,
    quantization_method=QUANT_METHOD,
)

import pathlib
gguf_files = list(pathlib.Path(GGUF_OUTPUT_DIR).glob("*.gguf"))
for f in gguf_files:
    size_gb = f.stat().st_size / 1e9
    print(f"GGUF: {f.name}  ({size_gb:.2f} GB)")

### Create the Ollama Modelfile

An Ollama `Modelfile` tells Ollama how to serve the GGUF — system prompt, temperature,
context size, and stop tokens.

Refs:
- Ollama Modelfile spec: https://github.com/ollama/ollama/blob/main/docs/modelfile.md
- Unsloth → Ollama guide: https://docs.unsloth.ai/basics/saving-models#ollama

In [ ]:
import pathlib

gguf_path = pathlib.Path(GGUF_OUTPUT_DIR)
gguf_file = next(gguf_path.glob("*.gguf"), None)

if gguf_file is None:
    print("⚠️  No GGUF file found — run the export cell above first.")
else:
    modelfile_content = f"""FROM ./{gguf_file.name}

PARAMETER temperature 0.7
PARAMETER num_ctx 4096
PARAMETER num_predict 1024
PARAMETER stop "<end_of_turn>"
PARAMETER stop "<start_of_turn>"

SYSTEM \"\"\"
You are Concert Med Ops AI, a clinical decision support assistant for supervising physicians,
paramedics, and harm reduction volunteers at live music events and EDM festivals. You provide
rapid, actionable guidance on toxicology, overdose management, triage, and festival-specific
emergencies. Always prioritise patient safety.

For life-threatening emergencies, direct the user to activate EMS (call 911) immediately.

\u26a0\ufe0f DISCLAIMER: AI-generated guidance. Not a substitute for licensed medical care.
Final treatment and transport decisions rest with the supervising physician.
\"\"\"
"""
    modelfile_path = gguf_path / "Modelfile"
    modelfile_path.write_text(modelfile_content)
    print(f"Modelfile written: {modelfile_path}")
    print("\nNext steps to register with Ollama:")
    print(f"  cd {gguf_path.resolve()}")
    print(f"  ollama create concert-med-tox -f Modelfile")
    print(f"  ollama run concert-med-tox\n")
    print("Then set in Concert Med Ops .env:")
    print("  MODEL_MEDICAL=concert-med-tox")

### Register with local Ollama (optional)

If Ollama is installed in this environment, you can register the model immediately.

In [ ]:
import subprocess

# Uncomment to register with a local Ollama instance
# result = subprocess.run(
#     ["ollama", "create", "concert-med-tox", "-f", str(modelfile_path)],
#     capture_output=True, text=True, cwd=str(gguf_path.resolve())
# )
# print(result.stdout)
# if result.returncode != 0:
#     print("ERROR:", result.stderr)

print("Skipped — uncomment the block above when running with a local Ollama.")

## 8. (Optional) Push LoRA Adapter to Hugging Face Hub

Push only the lightweight LoRA adapter weights (~50 MB) rather than the merged model.
Users then apply them on top of the base `gemma-4-E2B-it` at inference time.

Ref: https://docs.unsloth.ai/basics/saving-models#lora-adapters-uploading-to-hugging-face

> Create a HuggingFace token at https://huggingface.co/settings/tokens and set it as
> the `HF_TOKEN` environment variable or paste it in the `token` argument below.
>
> Repo name convention: `your-hf-username/concert-med-tox-gemma3-4b`

In [ ]:
import os

HF_TOKEN = os.environ.get("HF_TOKEN", "")  # set your HF write token here
HF_REPO = "YOUR_HF_USERNAME/concert-med-tox-gemma3-4b"  # change this

if not HF_TOKEN:
    print("⚠️  HF_TOKEN not set — skipping HuggingFace Hub push.")
    print("   Set HF_TOKEN environment variable and re-run this cell.")
else:
    # Push LoRA adapter only (small, ~50 MB)
    # Ref: https://docs.unsloth.ai/basics/saving-models#lora-adapters-uploading-to-hugging-face
    model.push_to_hub(HF_REPO, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
    print(f"Pushed LoRA adapter to: https://huggingface.co/{HF_REPO}")

    # Or push the merged GGUF (larger, but standalone)
    # model.push_to_hub_gguf(HF_REPO, tokenizer, quantization_method=QUANT_METHOD, token=HF_TOKEN)

## Appendix A — Cloud Serving (Approach A: Ollama on Cloud Run)

Once the GGUF is ready, deploy it to Google Cloud Run so the Concert Med Ops backend
can call it remotely when `CLOUD_MODE=true`.

Ref: `cloud/ollama-server/Dockerfile` in this repo.

```bash
# Build and push the Cloud Run image
gcloud builds submit cloud/ollama-server \
  --tag gcr.io/$PROJECT_ID/concert-med-ollama

# Deploy to Cloud Run (GPU L4)
gcloud run deploy concert-med-ollama \
  --image gcr.io/$PROJECT_ID/concert-med-ollama \
  --region us-central1 \
  --gpu 1 --gpu-type nvidia-l4 \
  --memory 16Gi --cpu 4 \
  --port 11434 \
  --no-allow-unauthenticated
```

Then set in Concert Med Ops `.env`:
```
CLOUD_MODE=true
OLLAMA_HOST=https://concert-med-ollama-xxxx-uc.a.run.app
MODEL_MEDICAL=concert-med-tox
```

The existing `OllamaRouter` in `backend/ai/ollama_client.py` works unmodified —
it just speaks to the remote Ollama instead of localhost.

## Appendix B — Resuming from a Checkpoint

If training was interrupted, resume from the last checkpoint:

```python
trainer.train(resume_from_checkpoint=True)
```

Or specify a path:
```python
trainer.train(resume_from_checkpoint="concert-med-tox-checkpoints/checkpoint-60")
```

Ref: https://huggingface.co/docs/transformers/main_classes/trainer#resume-from-checkpoint

## Appendix C — Expanding the Training Set

Add more training examples by:

1. Adding chunks to `backend/data/harm_reduction_kb.json`
2. Adding Q&A templates to `QA_TEMPLATES` in `notebooks/generate_training_data.py`
3. Adding hand-crafted pairs to `HANDCRAFTED_PAIRS` in the same file
4. Re-running `build_dataset()` and re-running this notebook

Recommended additional KB sources for future iterations:
- SAMHSA TIPs (Treatment Improvement Protocols): https://store.samhsa.gov/collection/treatment-improvement-protocols
- DanceSafe drug information: https://dancesafe.org/drug-information/
- ACEP Clinical Policies for toxicology: https://www.acep.org/clinical-and-practice-management/clinical-policies/
- EMCrit toxicology protocols: https://emcrit.org/toxicology/
- UpToDate clinical overviews (requires licence)
- Tintinalli's Emergency Medicine (7th+ ed.) — Chapter 183: Toxicology